# 09 - Campaign Prediction
SMOTE-augmented ML pipeline for campaign conversion prediction.

## Campaign ML -- Note on Data Size
Original dataset: 100 rows. Standard train/test split not viable. Strategy: Apply SMOTE oversampling to reach 500+ balanced samples, then evaluate using StratifiedKFold (5 folds). All results represent cross-validated performance on synthetic-augmented data.

In [1]:
import pandas as pd, numpy as np, joblib, os
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import plotly.express as px, plotly.graph_objects as go
import warnings; warnings.filterwarnings('ignore')
SEED=42; np.random.seed(SEED)
PRIMARY='#635BFF'; RISK='#E74C3C'; SAFE='#27AE60'; NEUTRAL='#3498DB'; WARNING='#F39C12'; TEMPLATE='plotly_white'


In [2]:
df = pd.read_csv('data/processed/campaign_clean.csv')
print(f"Shape: {df.shape}")
# Feature engineering
df['contacted_before'] = (df['pdays'] != -1).astype(int)
df['campaign_intensity'] = df['campaign'] + df['previous']

# Encode categoricals
cat_cols = ['job','marital','education','default','housing','loan','contact','month','poutcome']
label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

target = 'y_binary'
if target not in df.columns:
    df[target] = (df['y'] == 'yes').astype(int)

feature_cols = [c for c in df.columns if c not in ['y', target, 'index']]
X = df[feature_cols].select_dtypes(include=[np.number])
y = df[target]
print(f"Features: {X.shape}, Target dist:\n{y.value_counts()}")


Shape: (100, 23)
Features: (100, 21), Target dist:
y_binary
0    97
1     3
Name: count, dtype: int64


In [3]:
# SMOTE
smote = SMOTE(random_state=SEED, k_neighbors=2)
X_res, y_res = smote.fit_resample(X, y)
print(f"Original shape: {X.shape}, Resampled: {X_res.shape}")
print(f"Resampled target dist:\n{y_res.value_counts()}")


Original shape: (100, 21), Resampled: (194, 21)
Resampled target dist:
y_binary
0    97
1    97
Name: count, dtype: int64


In [4]:
# Cross-validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
models_cv = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=SEED),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=SEED),
    'XGBoost': XGBClassifier(n_estimators=100, random_state=SEED, eval_metric='auc', use_label_encoder=False)
}
cv_results = []
for name, model in models_cv.items():
    acc_scores = cross_val_score(model, X_res, y_res, cv=skf, scoring='accuracy')
    f1_scores = cross_val_score(model, X_res, y_res, cv=skf, scoring='f1')
    auc_scores = cross_val_score(model, X_res, y_res, cv=skf, scoring='roc_auc')
    print(f"{name}: Acc={acc_scores.mean():.4f}+/-{acc_scores.std():.4f}, "
          f"F1={f1_scores.mean():.4f}+/-{f1_scores.std():.4f}, "
          f"AUC={auc_scores.mean():.4f}+/-{auc_scores.std():.4f}")
    cv_results.append({'Model': name, 'Accuracy': acc_scores.mean(), 'F1': f1_scores.mean(),
                       'ROC-AUC': auc_scores.mean(), 'AUC_std': auc_scores.std()})

cv_df = pd.DataFrame(cv_results)
best_name = cv_df.loc[cv_df['ROC-AUC'].idxmax(), 'Model']
print(f"\nBest model: {best_name}")


Logistic Regression: Acc=0.9949+/-0.0103, F1=0.9949+/-0.0103, AUC=0.9900+/-0.0200


Random Forest: Acc=0.9949+/-0.0103, F1=0.9949+/-0.0103, AUC=1.0000+/-0.0000


XGBoost: Acc=0.9897+/-0.0126, F1=0.9897+/-0.0126, AUC=0.9858+/-0.0186

Best model: Random Forest


In [5]:
# Fit best model on full resampled data
best_model = models_cv[best_name]
best_model.fit(X_res, y_res)
print(f"Best model ({best_name}) trained on full resampled data.")


Best model (Random Forest) trained on full resampled data.


In [6]:
# Plot a: y distribution before/after SMOTE
before = y.value_counts().reset_index(); before.columns=['class','count']; before['stage']='Before SMOTE'
after = y_res.value_counts().reset_index(); after.columns=['class','count']; after['stage']='After SMOTE'
combined = pd.concat([before, after])
combined['class'] = combined['class'].astype(str)
fig = px.bar(combined, x='class', y='count', color='stage', barmode='group',
             color_discrete_sequence=[RISK, SAFE], template=TEMPLATE,
             title='Target Distribution: Before vs After SMOTE')
fig.show()


In [7]:
# Plot b-d: Conversion rates
df_orig = pd.read_csv('data/processed/campaign_clean.csv')
if 'y_binary' not in df_orig.columns: df_orig['y_binary'] = (df_orig['y']=='yes').astype(int)

conv_job = df_orig.groupby('job')['y_binary'].mean().sort_values(ascending=True).reset_index()
fig = px.bar(conv_job, y='job', x='y_binary', orientation='h', color_discrete_sequence=[PRIMARY],
             template=TEMPLATE, title='Conversion Rate by Job')
fig.show()

conv_contact = df_orig.groupby('contact')['y_binary'].mean().reset_index()
fig = px.bar(conv_contact, x='contact', y='y_binary', color_discrete_sequence=[SAFE],
             template=TEMPLATE, title='Conversion Rate by Contact')
fig.show()

conv_month = df_orig.groupby('month')['y_binary'].mean().reset_index()
fig = px.bar(conv_month, x='month', y='y_binary', color_discrete_sequence=[WARNING],
             template=TEMPLATE, title='Conversion Rate by Month')
fig.show()


In [8]:
# Plot e: campaign_intensity vs conversion
df_orig['campaign_intensity'] = df_orig['campaign'] + df_orig['previous']
ci = df_orig.groupby('campaign_intensity')['y_binary'].mean().reset_index()
fig = px.line(ci, x='campaign_intensity', y='y_binary', markers=True,
              color_discrete_sequence=[PRIMARY], template=TEMPLATE,
              title='Campaign Intensity vs Conversion Rate')
fig.show()


In [9]:
# Plot f: Feature importance
if hasattr(best_model, 'feature_importances_'):
    imp = pd.DataFrame({'feature': X.columns, 'importance': best_model.feature_importances_})
    imp = imp.sort_values('importance', ascending=True).tail(15)
    fig = px.bar(imp, y='feature', x='importance', orientation='h',
                 color_discrete_sequence=[PRIMARY], template=TEMPLATE,
                 title=f'Feature Importance - {best_name}')
    fig.show()


In [10]:
# Save
os.makedirs('models/campaign', exist_ok=True)
joblib.dump(best_model, 'models/campaign/xgboost_campaign.pkl')
print("Saved campaign model")

df[list(X.columns) + [target]].to_parquet('data/features/campaign_features.parquet', index=False)
print("Saved campaign_features.parquet")


Saved campaign model
Saved campaign_features.parquet
